[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/state-schema.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239426-lesson-1-state-schema)

# 状态 Schema

## 回顾

在模块 1 中，我们打下了基础！我们构建了一个 Agent，它可以：

* `act` - 让模型调用特定的工具
* `observe` - 将工具输出传回模型
* `reason` - 让模型根据工具输出进行推理，决定下一步做什么（例如，调用另一个工具或直接回复）
* `persist state` - 使用内存检查点来支持带有中断的长时间对话

我们还展示了如何在 LangGraph Studio 中本地运行它，或使用 LangGraph Cloud 部署它。

## 目标

在本模块中，我们将深入理解状态和记忆。

首先，让我们回顾几种不同的定义状态 Schema 的方式。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph

## Schema

当我们定义 LangGraph 的 `StateGraph` 时，需要使用一个[状态 Schema](https://docs.langchain.com/oss/python/langgraph/graph-api/#state)。

状态 Schema 表示图将使用的数据的结构和类型。

所有节点都需要按照该 Schema 进行通信。

LangGraph 在定义状态 Schema 方面提供了灵活性，支持各种 Python [类型](https://docs.python.org/3/library/stdtypes.html#type-objects)和验证方式！

## TypedDict

正如模块 1 中提到的，我们可以使用 Python 的 `typing` 模块中的 `TypedDict` 类。

它允许你指定键及其对应的值类型。

但请注意，这些只是类型提示。

它们可以被静态类型检查器（如 [mypy](https://github.com/python/mypy)）或 IDE 使用，在代码运行之前捕获潜在的类型相关错误。

但它们在运行时不会被强制执行！

In [ ]:
from typing_extensions import TypedDict

class TypedDictState(TypedDict):
    foo: str
    bar: str

对于更具体的值约束，你可以使用 `Literal` 类型提示。

这里，`mood` 只能是 "happy" 或 "sad"。

In [ ]:
from typing import Literal

class TypedDictState(TypedDict):
    name: str
    mood: Literal["happy","sad"]

我们可以通过将定义的状态类（例如 `TypedDictState`）传递给 `StateGraph` 来在 LangGraph 中使用它。

我们可以将每个状态键视为图中的一个"通道"。

正如模块 1 中所讨论的，我们在每个节点中覆盖指定键或"通道"的值。

In [ ]:
import random
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

def node_1(state):
    print("---Node 1---")
    return {"name": state['name'] + " is ... "}

def node_2(state):
    print("---Node 2---")
    return {"mood": "happy"}

def node_3(state):
    print("---Node 3---")
    return {"mood": "sad"}

def decide_mood(state) -> Literal["node_2", "node_3"]:
        
    # 这里我们在节点 2 和 3 之间做 50/50 的随机分配
    if random.random() < 0.5:

        # 50% 的概率返回节点 2
        return "node_2"
    
    # 50% 的概率返回节点 3
    return "node_3"

# 构建图
builder = StateGraph(TypedDictState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_mood)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

因为我们的状态是一个字典，所以我们可以直接使用字典来调用图，为状态中的 `name` 键设置初始值。

In [ ]:
graph.invoke({"name":"Lance"})

## Dataclass

Python 的 [dataclasses](https://docs.python.org/3/library/dataclasses.html) 提供了[另一种定义结构化数据的方式](https://www.datacamp.com/tutorial/python-data-classes)。

Dataclass 提供了简洁的语法来创建主要用于存储数据的类。

In [ ]:
from dataclasses import dataclass

@dataclass
class DataclassState:
    name: str
    mood: Literal["happy","sad"]

要访问 `dataclass` 的键，我们只需要修改 `node_1` 中使用的下标方式：

* 对于 `dataclass` 状态，我们使用 `state.name`，而上面对 `TypedDict` 则使用 `state["name"]`

你会注意到一个有点奇怪的地方：在每个节点中，我们仍然返回一个字典来执行状态更新。

这是可行的，因为 LangGraph 将状态对象的每个键分开存储。

节点返回的对象只需要拥有与状态中匹配的键（属性）！

在这种情况下，`dataclass` 有键 `name`，所以我们可以通过从节点返回字典来更新它，就像状态是 `TypedDict` 时那样。

In [ ]:
def node_1(state):
    print("---Node 1---")
    return {"name": state.name + " is ... "}

# 构建图
builder = StateGraph(DataclassState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_mood)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

我们使用 `dataclass` 来调用图，为状态中的每个键/通道设置初始值！

In [ ]:
graph.invoke(DataclassState(name="Lance",mood="sad"))

## Pydantic

如前所述，`TypedDict` 和 `dataclasses` 提供了类型提示，但它们不会在运行时强制执行类型。

这意味着你可能会分配无效的值而不会引发错误！

例如，我们可以将 `mood` 设置为 `mad`，即使我们的类型提示指定了 `mood: list[Literal["happy","sad"]]`。

In [ ]:
dataclass_instance = DataclassState(name="Lance", mood="mad")

[Pydantic](https://docs.pydantic.dev/latest/api/base_model/) 是一个使用 Python 类型注解的数据验证和设置管理库。

由于其验证能力，它特别适合[在 LangGraph 中定义状态 Schema](https://docs.langchain.com/oss/python/langgraph/use-graph-api#use-pydantic-models-for-graph-state)。

Pydantic 可以在运行时执行验证，检查数据是否符合指定的类型和约束。

In [ ]:
from pydantic import BaseModel, field_validator, ValidationError

class PydanticState(BaseModel):
    name: str
    mood: str # "happy" 或 "sad"

    @field_validator('mood')
    @classmethod
    def validate_mood(cls, value):
        # 确保 mood 是 "happy" 或 "sad"
        if value not in ["happy", "sad"]:
            raise ValueError("mood 必须是 'happy' 或 'sad'")
        return value

try:
    state = PydanticState(name="John Doe", mood="mad")
except ValidationError as e:
    print("验证错误:", e)

我们可以在图中无缝使用 `PydanticState`。

In [ ]:
# 构建图
builder = StateGraph(PydanticState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_mood)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke(PydanticState(name="Lance",mood="sad"))